# Python Memory & Performance
Covers: sys.getsizeof, tracemalloc, gc module, reference counting, weak references, __slots__, generators vs lists, timeit, cProfile

## 1. sys.getsizeof — Object Sizes

In [ ]:
import sys

# Basic object sizes
objects = [
    ('None', None),
    ('True', True),
    ('int(0)', 0),
    ('int(42)', 42),
    ('int(10**100)', 10**100),
    ('float', 3.14),
    ('str empty', ''),
    ('str hello', 'hello'),
    ('list empty', []),
    ('list [1,2,3]', [1, 2, 3]),
    ('dict empty', {}),
    ('dict {a:1}', {'a': 1}),
    ('tuple ()', ()),
    ('tuple (1,2)', (1, 2)),
    ('set empty', set()),
]

print(f'{'Object':<20} {'Size (bytes)':>12}')
print('-' * 34)
for name, obj in objects:
    print(f'{name:<20} {sys.getsizeof(obj):>12}')

# getsizeof doesn't include referenced objects!
lst = [1, 2, 3, 4, 5]
print(f'\nList [1..5] shallow: {sys.getsizeof(lst)} bytes')
print(f'Each int: {sys.getsizeof(1)} bytes')
print(f'True total: {sys.getsizeof(lst) + sum(sys.getsizeof(x) for x in lst)} bytes')

## 2. tracemalloc — Memory Tracing

In [ ]:
import tracemalloc

# Basic usage
tracemalloc.start()

# Allocate some memory
data = {i: [j**2 for j in range(100)] for i in range(100)}

current, peak = tracemalloc.get_traced_memory()
print(f'Current: {current / 1024:.1f} KB')
print(f'Peak: {peak / 1024:.1f} KB')

snapshot = tracemalloc.take_snapshot()
tracemalloc.stop()

# Top allocations
top_stats = snapshot.statistics('lineno')
print('\nTop 5 allocations:')
for stat in top_stats[:5]:
    print(f'  {stat}')

# Compare snapshots
tracemalloc.start()
snap1 = tracemalloc.take_snapshot()

# Allocate more
more_data = [list(range(1000)) for _ in range(50)]

snap2 = tracemalloc.take_snapshot()
tracemalloc.stop()

diff = snap2.compare_to(snap1, 'lineno')
print('\nMemory increase (top 3):')
for stat in diff[:3]:
    print(f'  {stat}')

## 3. gc Module — Garbage Collection

In [ ]:
import gc
import sys

print('GC enabled:', gc.isenabled())
print('GC thresholds:', gc.get_threshold())
print('GC counts:', gc.get_count())

# Reference counting demo
class Tracked:
    def __init__(self, name):
        self.name = name
    def __del__(self):
        print(f'  Deleted: {self.name}')

print('\n--- Reference counting ---')
obj = Tracked('obj1')
print(f'refcount(obj): {sys.getrefcount(obj)}')
ref2 = obj
print(f'refcount after ref2=obj: {sys.getrefcount(obj)}')
del ref2
print(f'refcount after del ref2: {sys.getrefcount(obj)}')
print('Deleting obj...')
del obj  # refcount → 0 → immediate deletion

# Reference cycle
print('\n--- Reference cycle ---')
class Node:
    def __init__(self, name):
        self.name = name
        self.ref = None
    def __del__(self):
        print(f'  Deleted: {self.name}')

a = Node('A')
b = Node('B')
a.ref = b  # A → B
b.ref = a  # B → A (cycle!)

print('Deleting a and b (cycle — not immediately freed)...')
del a, b  # refcount > 0 due to cycle — NOT immediately freed

print('Forcing gc.collect()...')
collected = gc.collect()  # cycle collector finds and frees them
print(f'Collected: {collected} objects')

## 4. Weak References

In [ ]:
import weakref
import gc

class Resource:
    def __init__(self, name):
        self.name = name
    def __repr__(self):
        return f'Resource({self.name!r})'

# Weak reference — doesn't prevent GC
obj = Resource('important')
weak = weakref.ref(obj)

print('Object:', obj)
print('Weak ref:', weak())  # access via ()
print('Is alive:', weak() is not None)

del obj  # refcount → 0 → deleted
gc.collect()
print('After del:', weak())  # None
print('Is alive:', weak() is not None)

# WeakValueDictionary — cache that doesn't prevent GC
print('\n--- WeakValueDictionary cache ---')
cache = weakref.WeakValueDictionary()

def get_resource(name):
    if name in cache:
        print(f'  Cache hit: {name}')
        return cache[name]
    print(f'  Cache miss: creating {name}')
    res = Resource(name)
    cache[name] = res
    return res

r1 = get_resource('db_conn')
r2 = get_resource('db_conn')  # cache hit
print('Same object:', r1 is r2)
print('Cache size:', len(cache))

del r1, r2  # no more strong references
gc.collect()
print('Cache size after del:', len(cache))  # 0 — auto-removed!

# Callback on weak reference death
def on_finalize(ref):
    print(f'  Object finalized!')

obj = Resource('temp')
weak = weakref.ref(obj, on_finalize)
del obj

## 5. Generators vs Lists — Memory Efficiency

In [ ]:
import sys
import tracemalloc
from itertools import islice

N = 1_000_000

# List — all in memory
tracemalloc.start()
lst = [x**2 for x in range(N)]
_, list_peak = tracemalloc.get_traced_memory()
tracemalloc.stop()

# Generator — lazy, O(1) memory
tracemalloc.start()
gen = (x**2 for x in range(N))
_, gen_peak = tracemalloc.get_traced_memory()
tracemalloc.stop()

print(f'List ({N:,} items): {list_peak / 1024 / 1024:.1f} MB')
print(f'Generator:          {gen_peak} bytes')
print(f'Ratio: {list_peak / gen_peak:.0f}x')

# Generator pipeline — process large data with O(1) memory
def read_numbers(n):
    """Simulate reading from a large file."""
    for i in range(n):
        yield i

def filter_even(numbers):
    for n in numbers:
        if n % 2 == 0:
            yield n

def square(numbers):
    for n in numbers:
        yield n ** 2

# Pipeline — each stage processes one item at a time
pipeline = square(filter_even(read_numbers(N)))
total = sum(islice(pipeline, 1000))  # only process first 1000
print(f'\nPipeline sum (first 1000 even squares): {total}')

# Infinite generator
def fibonacci():
    a, b = 0, 1
    while True:
        yield a
        a, b = b, a + b

fibs = list(islice(fibonacci(), 15))
print(f'First 15 Fibonacci: {fibs}')

## 6. timeit and cProfile — Benchmarking

In [ ]:
import timeit
import cProfile
import pstats
import io

# timeit — micro-benchmarks
print('=== timeit benchmarks ===')

# List comprehension vs map
t1 = timeit.timeit('[x**2 for x in range(1000)]', number=10000)
t2 = timeit.timeit('list(map(lambda x: x**2, range(1000)))', number=10000)
t3 = timeit.timeit('list(map(pow, range(1000), [2]*1000))', number=10000)
print(f'List comprehension: {t1:.3f}s')
print(f'map + lambda:       {t2:.3f}s')
print(f'map + pow:          {t3:.3f}s')

# Membership testing
setup = 'data = list(range(10000)); data_set = set(data); data_dict = dict.fromkeys(data)'
t_list = timeit.timeit('5000 in data', setup=setup, number=100000)
t_set = timeit.timeit('5000 in data_set', setup=setup, number=100000)
t_dict = timeit.timeit('5000 in data_dict', setup=setup, number=100000)
print(f'\nMembership test (10k items):')
print(f'  list: {t_list:.3f}s')
print(f'  set:  {t_set:.3f}s ({t_list/t_set:.0f}x faster)')
print(f'  dict: {t_dict:.3f}s ({t_list/t_dict:.0f}x faster)')

# String concatenation
t_concat = timeit.timeit(
    'result = ""; [result := result + s for s in strings]',
    setup='strings = [str(i) for i in range(100)]',
    number=1000
)
t_join = timeit.timeit(
    '"".join(strings)',
    setup='strings = [str(i) for i in range(100)]',
    number=1000
)
print(f'\nString building (100 strings):')
print(f'  += concat: {t_concat:.4f}s')
print(f'  join:      {t_join:.4f}s ({t_concat/t_join:.0f}x faster)')

In [ ]:
import cProfile
import pstats
import io

# cProfile — find bottlenecks
def slow_sort(data):
    """Bubble sort — O(n^2)."""
    data = list(data)
    n = len(data)
    for i in range(n):
        for j in range(0, n-i-1):
            if data[j] > data[j+1]:
                data[j], data[j+1] = data[j+1], data[j]
    return data

def fast_sort(data):
    """Python's built-in sort — O(n log n)."""
    return sorted(data)

def workload():
    import random
    data = [random.randint(1, 1000) for _ in range(500)]
    slow_sort(data)
    fast_sort(data)

# Profile
pr = cProfile.Profile()
pr.enable()
for _ in range(5):
    workload()
pr.disable()

# Print stats
stream = io.StringIO()
ps = pstats.Stats(pr, stream=stream)
ps.sort_stats('cumulative')
ps.print_stats(8)
print(stream.getvalue())